# Agentic Multi-Model Platform — Jupyter Live Lab

This notebook is the executable learning/prototype edition of the Azure + MLflow repository. It demonstrates the complete path:

**question + data → task formulation → algorithm selection → heterogeneous training → MLflow registration → champion/challenger aliases → normalized comparison → relay-agent tool response**

The notebook runs locally, but its boundaries correspond directly to the production services: FastAPI/MCP control plane, MLflow lifecycle layer, and dependency-isolated model-serving pools on AKS.

## 1. Architecture

The notebook keeps everything in one kernel for learning. Production separates these into independently deployable services.

```text
Browser/API client ──REST/SSE──┐
                               ├─> Shared orchestrator ─> MLflow Registry
Relay LLM ────────MCP tools────┘           │
                                           ├─> sklearn/XGBoost CPU pool
                                           ├─> PyTorch GPU pool
                                           └─> TensorFlow GPU pool
```

MCP is the relay agent's tool interface. REST is the browser/programmatic interface. Neither contains the model framework dependencies.

In [ ]:
# Run once if the notebook environment does not already contain these packages.
%pip install -q mlflow scikit-learn pandas numpy matplotlib pydantic fastapi httpx

In [ ]:
from __future__ import annotations

import json
import time
from dataclasses import asdict, dataclass
from pathlib import Path
from typing import Any

import matplotlib.pyplot as plt
import mlflow
import mlflow.sklearn
import numpy as np
import pandas as pd
from mlflow import MlflowClient
from mlflow.models import infer_signature
from sklearn.base import clone
from sklearn.datasets import make_classification
from sklearn.ensemble import HistGradientBoostingClassifier, RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import average_precision_score, brier_score_loss, f1_score, recall_score
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import GaussianNB
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

RANDOM_STATE = 42
pd.set_option('display.max_colwidth', 120)

## 2. Algorithm catalog — the relay agent's Rolodex

Frameworks such as scikit-learn and PyTorch are tooling/runtime metadata. Algorithms such as KNN, logistic regression, and random forest are the candidates selected for a task.

In [ ]:
ALGORITHM_CATALOG = pd.DataFrame([
    {'algorithm': 'logistic_regression', 'family': 'linear', 'framework': 'scikit-learn', 'task': 'binary_classification', 'purpose': 'interpretable baseline'},
    {'algorithm': 'knn', 'family': 'nearest_neighbor', 'framework': 'scikit-learn', 'task': 'binary_classification', 'purpose': 'local similarity'},
    {'algorithm': 'naive_bayes', 'family': 'probabilistic', 'framework': 'scikit-learn', 'task': 'binary_classification', 'purpose': 'fast probabilistic baseline'},
    {'algorithm': 'random_forest', 'family': 'tree_ensemble', 'framework': 'scikit-learn', 'task': 'binary_classification', 'purpose': 'nonlinear bagged trees'},
    {'algorithm': 'gradient_boosting', 'family': 'boosted_tree', 'framework': 'scikit-learn', 'task': 'binary_classification', 'purpose': 'high-performance tabular candidate'},
])
ALGORITHM_CATALOG

## 3. Question and governed dataset

Replace the sample question and dataset later. The synthetic dataset represents customer-churn features and deliberately contains class imbalance.

In [ ]:
QUESTION = 'Which customers are likely to cancel next month?'
TARGET = 'churn_within_30_days'

X_array, y_array = make_classification(
    n_samples=5000, n_features=16, n_informative=9, n_redundant=3,
    weights=[0.82, 0.18], class_sep=1.1, random_state=RANDOM_STATE,
)
feature_names = [f'feature_{i:02d}' for i in range(X_array.shape[1])]
X = pd.DataFrame(X_array, columns=feature_names)
y = pd.Series(y_array, name=TARGET)
X_train, X_valid, y_train, y_valid = train_test_split(
    X, y, test_size=0.25, stratify=y, random_state=RANDOM_STATE
)
{'rows': len(X), 'features': X.shape[1], 'positive_rate': round(float(y.mean()), 4)}

In [ ]:
def formulate_ml_problem(question: str, target: str | None, rows: int) -> dict[str, Any]:
    """Deterministic safety layer around the relay LLM's proposed task."""
    text = question.lower()
    if any(word in text for word in ('likely', 'whether', 'cancel', 'fail')):
        task = 'binary_classification'
    elif any(word in text for word in ('forecast', 'time series', 'future values')):
        task = 'forecasting'
    elif any(word in text for word in ('how much', 'price', 'amount')):
        task = 'regression'
    else:
        raise ValueError('Clarification is required before selecting algorithms.')
    return {
        'question': question, 'task': task, 'target': target, 'rows': rows,
        'metrics': ['pr_auc', 'recall', 'f1', 'brier_score'],
    }

problem = formulate_ml_problem(QUESTION, TARGET, len(X))
problem

## 4. Select heterogeneous candidate algorithms

This is the tool-backed model-selection step. In production it also checks MLflow signatures, deployed aliases, latency, runtime class, dataset size, and security policy.

In [ ]:
def recommend_algorithms(problem: dict[str, Any], maximum_models: int = 5) -> pd.DataFrame:
    candidates = ALGORITHM_CATALOG[ALGORITHM_CATALOG.task == problem['task']].copy()
    if problem['rows'] > 100_000:
        candidates = candidates[candidates.algorithm != 'knn']
    return candidates.head(maximum_models)

selected = recommend_algorithms(problem)
selected

## 5. Train and evaluate the candidates

Every algorithm receives the same governed train/validation split. Algorithm-specific preprocessing remains inside its pipeline so the artifact can carry preprocessing to its endpoint.

In [ ]:
MODEL_FACTORIES = {
    'logistic_regression': make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000)),
    'knn': make_pipeline(StandardScaler(), KNeighborsClassifier(n_neighbors=15, weights='distance')),
    'naive_bayes': GaussianNB(),
    'random_forest': RandomForestClassifier(n_estimators=300, min_samples_leaf=3, n_jobs=-1, random_state=RANDOM_STATE),
    'gradient_boosting': HistGradientBoostingClassifier(max_iter=200, random_state=RANDOM_STATE),
}

@dataclass
class CandidateResult:
    algorithm: str
    model: Any
    pr_auc: float
    recall: float
    f1: float
    brier_score: float
    latency_ms: float
    probabilities: np.ndarray

results: list[CandidateResult] = []
for algorithm in selected.algorithm:
    model = clone(MODEL_FACTORIES[algorithm])
    model.fit(X_train, y_train)
    started = time.perf_counter()
    probability = model.predict_proba(X_valid)[:, 1]
    latency_ms = (time.perf_counter() - started) * 1000
    prediction = (probability >= 0.5).astype(int)
    results.append(CandidateResult(
        algorithm=algorithm, model=model,
        pr_auc=average_precision_score(y_valid, probability),
        recall=recall_score(y_valid, prediction),
        f1=f1_score(y_valid, prediction),
        brier_score=brier_score_loss(y_valid, probability),
        latency_ms=latency_ms, probabilities=probability,
    ))

leaderboard = pd.DataFrame([{k: v for k, v in asdict(r).items() if k not in {'model', 'probabilities'}} for r in results])
leaderboard.sort_values('pr_auc', ascending=False).reset_index(drop=True)

In [ ]:
plot_data = leaderboard.sort_values('pr_auc')
ax = plot_data.plot.barh(x='algorithm', y=['pr_auc', 'recall', 'f1'], figsize=(10, 5))
ax.set_title('Heterogeneous model comparison on one validation snapshot')
ax.set_xlim(0, 1)
ax.set_xlabel('Score')
plt.tight_layout();

## 6. Log, register, and alias models with MLflow

Each artifact carries a signature and dependency metadata. Production endpoints use dependency-isolated runtime images; the gateway does not install these ML packages.

In [ ]:
TRACKING_DB = Path('mlflow_notebook.db').resolve()
mlflow.set_tracking_uri(f'sqlite:///{TRACKING_DB.as_posix()}')
mlflow.set_experiment('agentic-multi-model-notebook')

registered_versions: dict[str, int] = {}
for item in results:
    registered_name = f'customer-churn-{item.algorithm.replace("_", "-")}'
    with mlflow.start_run(run_name=item.algorithm):
        mlflow.log_params({'algorithm': item.algorithm, 'task': problem['task'], 'runtime_class': 'tabular-cpu'})
        mlflow.log_metrics({
            'pr_auc': item.pr_auc, 'recall': item.recall, 'f1': item.f1,
            'brier_score': item.brier_score, 'latency_ms': item.latency_ms,
        })
        signature = infer_signature(X_train, item.model.predict_proba(X_train.iloc[:5]))
        info = mlflow.sklearn.log_model(
            sk_model=item.model, name='model', signature=signature,
            input_example=X_train.iloc[:2], registered_model_name=registered_name,
            extra_pip_requirements=['scikit-learn>=1.5,<2'],
        )
        versions = MlflowClient().search_model_versions(f"name='{registered_name}'")
        registered_versions[registered_name] = max(int(v.version) for v in versions)

registered_versions

In [ ]:
# Assign one within-algorithm champion alias per registered identity.
# Across algorithms, the best validation candidate becomes the task champion.
client = MlflowClient()
for model_name, version in registered_versions.items():
    client.set_registered_model_alias(model_name, 'champion', version)

best_algorithm = leaderboard.sort_values('pr_auc', ascending=False).iloc[0].algorithm
task_champion = f'customer-churn-{best_algorithm.replace("_", "-")}'
{
    'task': problem['task'],
    'task_champion': f'models:/{task_champion}@champion',
    'algorithm_champions': [f'models:/{name}@champion' for name in registered_versions],
}

## 7. Normalize predictions as microservice responses

The LLM receives structured JSON. Python—not the LLM—calculates agreement and the ensemble.

In [ ]:
sample = X_valid.iloc[[0]]

def invoke_model_service(item: CandidateResult, features: pd.DataFrame) -> dict[str, Any]:
    started = time.perf_counter()
    probability = float(item.model.predict_proba(features)[0, 1])
    return {
        'model_name': f'customer-churn-{item.algorithm}',
        'algorithm': item.algorithm,
        'framework': 'scikit-learn',
        'runtime_class': 'tabular-cpu',
        'task': 'binary_classification',
        'prediction': {'label': 'churn' if probability >= 0.5 else 'retain', 'probability': probability},
        'validation_metrics': {'pr_auc': item.pr_auc, 'recall': item.recall, 'brier_score': item.brier_score},
        'latency_ms': (time.perf_counter() - started) * 1000,
    }

service_results = [invoke_model_service(item, sample) for item in results]
pd.DataFrame([{
    'algorithm': r['algorithm'], 'label': r['prediction']['label'],
    'probability': r['prediction']['probability'], 'validation_pr_auc': r['validation_metrics']['pr_auc']
} for r in service_results])

In [ ]:
def deterministic_comparison(model_results: list[dict[str, Any]]) -> dict[str, Any]:
    # Weight by validation PR-AUC; this can be replaced by a calibrated stacking model.
    probabilities = np.array([r['prediction']['probability'] for r in model_results])
    weights = np.array([r['validation_metrics']['pr_auc'] for r in model_results])
    combined = float(np.average(probabilities, weights=weights))
    labels = [r['prediction']['label'] for r in model_results]
    majority = max(set(labels), key=labels.count)
    return {
        'models_executed': len(model_results),
        'agreement': labels.count(majority) / len(labels),
        'recommended_prediction': {
            'label': 'churn' if combined >= 0.5 else 'retain',
            'probability': combined,
        },
        'basis': 'validation-PR-AUC-weighted heterogeneous ensemble',
        'model_results': model_results,
    }

comparison = deterministic_comparison(service_results)
comparison

## 8. Relay-agent tool contract

In production this function is exposed as the coarse-grained MCP tool `compare_compatible_models`. The agent calls one business-level tool; the orchestrator handles model versions and service locations.

In [ ]:
def compare_compatible_models_tool(
    question: str, data: pd.DataFrame, maximum_models: int = 3
) -> dict[str, Any]:
    formulated = formulate_ml_problem(question, TARGET, len(data))
    ranked = sorted(results, key=lambda item: item.pr_auc, reverse=True)[:maximum_models]
    model_outputs = [invoke_model_service(item, data) for item in ranked]
    return {
        'interpreted_problem': formulated,
        'execution_plan': [item.algorithm for item in ranked],
        'comparison': deterministic_comparison(model_outputs),
    }

tool_result = compare_compatible_models_tool(QUESTION, sample, maximum_models=3)
print(json.dumps(tool_result, indent=2))

In [ ]:
def relay_agent_response(tool_result: dict[str, Any]) -> str:
    """Deterministic stand-in for the final LLM narrative."""
    result = tool_result['comparison']['recommended_prediction']
    agreement = tool_result['comparison']['agreement']
    algorithms = ', '.join(tool_result['execution_plan'])
    return (
        f"The platform compared {algorithms}. The validation-weighted result is "
        f"{result['label']} with probability {result['probability']:.1%}. "
        f"Model agreement is {agreement:.0%}."
    )

relay_agent_response(tool_result)

## 9. Public endpoint map

The same orchestration function sits behind these public API Management routes:

```text
POST /api/v1/infer
POST /api/v1/tasks/binary_classification/compare
POST /api/v1/algorithms/knn/predict
POST /api/v1/algorithms/logistic_regression/predict
POST /api/v1/models/customer-churn-random-forest/predict
POST /api/v1/models/customer-churn-random-forest/versions/12/predict
```

The physical AKS model services remain private. API Management may provide friendly rewrites such as `/knn`, but the versioned routes are the durable contracts.

## 10. Production expansion checklist

1. Replace synthetic data with an approved dataset connector and immutable dataset version.
2. Split each runtime family into its dependency-isolated image.
3. Add XGBoost, PyTorch, TensorFlow, forecasting, and anomaly adapters.
4. Move analysis state to PostgreSQL and jobs to Azure Service Bus.
5. Put Azure API Management in front of FastAPI and use Entra ID.
6. Expose the comparison function through the repository's MCP server.
7. Add shadow/canary traffic and deterministic promotion gates.
8. Record API, MCP, and model spans in MLflow Tracing.
9. Require human approval before training, deployment, and alias promotion.